<div class='alert alert-warning'>

Интерактивные примеры NumPy — экспериментальная возможность, поэтому они не всегда работают как ожидается и могут долго загружаться, особенно на устройствах с ограниченными ресурсами. Версия NumPy в примерах также может отличаться от версии, для которой вы просматриваете документацию. Если возникнут проблемы, сообщите о них в [систему отслеживания ошибок NumPy](https://github.com/numpy/numpy/issues).

</div>

Вот как можно написать функцию ``iter_add`` с помощью протокола итераторов Python:


In [ ]:
import numpy as np

In [ ]:
def iter_add_py(x, y, out=None):
    addop = np.add
    it = np.nditer([x, y, out], [],
                [['readonly'], ['readonly'], ['writeonly','allocate']])
    with it:
        for (a, b, c) in it:
            addop(a, b, out=c)
        return it.operands[2]

Вот та же функция, но в соответствии с шаблоном C-стиля:

In [ ]:
def iter_add(x, y, out=None):
   addop = np.add
   it = np.nditer([x, y, out], [],
               [['readonly'], ['readonly'], ['writeonly','allocate']])
   with it:
       while not it.finished:
           addop(it[0], it[1], out=it[2])
           it.iternext()
       return it.operands[2]

Пример функции внешнего произведения:

In [ ]:
def outer_it(x, y, out=None):
    mulop = np.multiply
    it = np.nditer([x, y, out], ['external_loop'],
            [['readonly'], ['readonly'], ['writeonly', 'allocate']],
            op_axes=[list(range(x.ndim)) + [-1] * y.ndim,
                     [-1] * x.ndim + list(range(y.ndim)),
                     None])
    with it:
        for (a, b, c) in it:
            mulop(a, b, out=c)
        return it.operands[2]

In [ ]:
a = np.arange(2)+1
b = np.arange(3)+1
outer_it(a,b)

array([[1, 2, 3],
       [2, 4, 6]])

Пример функции, работающей как «лямбда»-ufunc:


In [ ]:
def luf(lamdaexpr, *args, **kwargs):
   '''luf(lambdaexpr, op1, ..., opn, out=None, order='K', casting='safe', buffersize=0)'''
   nargs = len(args)
   op = (kwargs.get('out',None),) + args
   it = np.nditer(op, ['buffered','external_loop'],
           [['writeonly','allocate','no_broadcast']] +
                           [['readonly','nbo','aligned']]*nargs,
           order=kwargs.get('order','K'),
           casting=kwargs.get('casting','safe'),
           buffersize=kwargs.get('buffersize',0))
   while not it.finished:
       it[0] = lamdaexpr(*it[1:])
       it.iternext()
   return it.operands[0]

In [ ]:
a = np.arange(5)
b = np.ones(5)
luf(lambda i,j:i*i + j/2, a, b)

array([  0.5,   1.5,   4.5,   9.5,  16.5])

При флагах операндов ``"writeonly"`` или ``"readwrite"`` операнды могут быть представлениями исходных данных с флагом `WRITEBACKIFCOPY`. В этом случае `nditer` нужно использовать как менеджер контекста либо вызвать метод `nditer.close` до использования результата. При работе с менеджером контекста временные данные записываются обратно в исходные при вызове метода `object.__exit__`, а не раньше:


In [ ]:
a = np.arange(6, dtype=np.int32)[::-2]
with np.nditer(a, [],
       [['writeonly', 'updateifcopy']],
       casting='unsafe',
       op_dtypes=[np.dtype(np.float32)]) as i:
   x = i.operands[0]
   x[:] = [-1, -2, -3]
   # a still unchanged here
a, x

(array([-1, -2, -3], dtype=int32), array([-1., -2., -3.], dtype=float32))

**Комментарии к примеру на русском**

Здесь a всё ещё не изменён

После выхода из итератора оставшиеся ссылки (например, `x` в примере) могут как сохранять общие данные с исходным массивом `a`, так и не сохранять их. Если была активна обратная запись, то есть `x.base.flags.writebackifcopy` равен `True`, выход из итератора разрывает связь между `x` и `a`: запись в `x` больше не изменяет `a`. Если обратная запись не была активна, `x.data` по-прежнему указывает на часть `a.data`, и изменение одного влияет на другой.

Поддержка контекстного менеджера и метод `close` появились в версии 1.15.0.